Extraindo PDF

In [84]:
!pip install transformers datasets pdfplumber accelerate sentencepiece PyMuPDF torch json re

ERROR: Could not find a version that satisfies the requirement json (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for json


In [85]:
import json
import re
# import pdfplumber
import fitz  # PyMuPDF
import re
import torch
from pathlib import Path
# from PyPDF2 import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

In [ ]:

def extract_text_from_pdf_pymupdf(pdf_path, skip_pages=6):
    """Extrai texto de um arquivo PDF usando PyMuPDF."""
    pages = []
    
    # Abre o documento PDF
    doc = fitz.open(pdf_path)
    
    # Itera pelas páginas a partir do índice 'skip_pages'
    for page_num in range(skip_pages, len(doc)):
        page = doc[page_num]
        page_text = page.get_text() # Extrai o texto da página

        if page_text:
            pages.append(page_text)
            
    doc.close() # É uma boa prática fechar o documento
    return "\n".join(pages)

def clean_pdf_text(text):
    # Corrige palavras quebradas por hífen no final da linha
    text = re.sub(r'(\w+)-\s+(\w+)', r'\1\2', text) 

    # Remove múltiplas quebras de linha
    text = re.sub(r'\n+', '\n', text)

    # Remove múltiplos espaços e tabulações
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()

# Configuração do caminho do PDF
pdf_path = "20260018701.pdf"

# Execução
full_text = extract_text_from_pdf_pymupdf(pdf_path)
full_text = clean_pdf_text(full_text)

print(f"Total de caracteres extraídos: {len(full_text)}")
print("\n--- INÍCIO DO TEXTO ---\n")
print(full_text[:500])

Total de caracteres extraídos: 264093

--- INÍCIO DO TEXTO ---

A medicina da simplicidade
– trabalhar com plantas é a
ciência do simples
Alésio dos Passos Santos – ambientalista, cultivador e educador de plantas medicinais (com contribuições de César Paulo
Simionatto, Murilo Leandro Marcos, Leila Nery dos Santos Souza,
Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon
Bittencourt)
A medicina do sinergismo e a energia de cura
O sinergismo é uma das coisas mais importantes no mundo
das plantas. O remédio de farmácia, o que que é? Um princípio
ativo i


Dividindo em chunks

In [60]:
def dividir_em_paragrafos(texto):
    """
    Divide o texto em parágrafos significativos.
    """
    # Dividindo os paragrafos por quebra de linha e limpando espaços extras
    paragrafos = texto.split("\n")
    # Filtro de ruído: ignora fragmentos menores que 50 caracteres (como números de página) 
    return [p.strip() for p in paragrafos if len(p.strip()) > 50]


In [61]:
paragrafos = dividir_em_paragrafos(full_text)
print(f"\nParágrafos encontrados: {len(paragrafos)}")


Parágrafos encontrados: 2867


In [62]:
def chunk_text_especializado(paragrafos, max_chunk_chars=500, overlap_chars=100):
    """
    Agrupa parágrafos em blocos menores (chunks) com sobreposição.
    
    Sugestões aplicadas:
    - max_chunk_chars=500: Chunks menores tendem a ser mais focados.
    - overlap_chars: Evita a perda de contexto nas 'emendas' entre blocos.
    """
    chunks = []
    chunk_atual = ""

    for paragrafo in paragrafos:
        # TRATAMENTO DE INTEGRIDADE: 
        # Se um parágrafo sozinho for maior que o limite máximo, ele é dividido em partes.
        if len(paragrafo) > max_chunk_chars:
            if chunk_atual:
                chunks.append(chunk_atual)
                chunk_atual = ""
            # Divide o parágrafo gigante respeitando o overlap
            for i in range(0, len(paragrafo), max_chunk_chars - overlap_chars):
                chunks.append(paragrafo[i : i + max_chunk_chars])
            continue

        # Lógica de agrupamento com verificação de limite
        if len(chunk_atual) + len(paragrafo) + 1 <= max_chunk_chars:
            chunk_atual += (" " if chunk_atual else "") + paragrafo 
        else:
            if chunk_atual:
                chunks.append(chunk_atual)
            
            # APLICAÇÃO DE SOBREPOSIÇÃO (Sliding Window):
            # O novo chunk começa com o final do chunk anterior para manter o contexto [2].
            inicio_com_contexto = chunk_atual[-overlap_chars:] if len(chunk_atual) > overlap_chars else ""
            chunk_atual = (inicio_com_contexto + " " if inicio_com_contexto else "") + paragrafo

    if chunk_atual:
        chunks.append(chunk_atual)

    return chunks


In [65]:
chunks = chunk_text_especializado(paragrafos, max_chunk_chars=600)
print(f"Número de chunks: {len(chunks)}")
print(f"\nExemplo de chunk (primeiro):\n{chunks[0][:600]}...")
print(f"\nExemplo de chunk (segundo):\n{chunks[1][:600]}...")

Número de chunks: 507

Exemplo de chunk (primeiro):
Alésio dos Passos Santos – ambientalista, cultivador e educador de plantas medicinais (com contribuições de César Paulo Simionatto, Murilo Leandro Marcos, Leila Nery dos Santos Souza, Claudete Espindola Tomaz, Clea Bregue Daniel e Paula Tonon O sinergismo é uma das coisas mais importantes no mundo das plantas. O remédio de farmácia, o que que é? Um princípio ativo isolado, concentrado, sintetizado, que faz o remédio. E a planta, o que que faz? Todos os princípios ativos juntos em sinergismo, trabalhando juntos, é a energia da planta que não se...

Exemplo de chunk (segundo):
odos os princípios ativos juntos em sinergismo, trabalhando juntos, é a energia da planta que não se vê em microscópio. Então pra mim, isolar o princípio ativo é perder o sinergismo da planta. O remédio do sinergismo é das coisas mais importantes nas plantas. Nunca vi livro com esse Não adianta estudar somente os grupos de princípios ativos. A planta tem que ter a

Carregando modelo

In [92]:
# --- Modelo Causal ---
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Modelo carregado com sucesso.")

Loading weights: 100%|██████████| 201/201 [00:04<00:00, 49.12it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Modelo carregado com sucesso.


In [103]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def is_bad_chunk(text, min_len=80):
    text = text.strip()
    if len(text) < min_len:
        return True
    words = text.split()
    if len(set(words)) < 10:
        return True
    strange_ratio = sum(1 for c in text if not c.isalnum() and c not in " .,;:-()[]{}") / max(len(text), 1) #
    if strange_ratio > 0.3:
        return True
    return False

def run_model_direct(model, tokenizer, prompt):
    inputs = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        inputs,
        max_new_tokens=256,   
        temperature=0.1,      # Temperatura baixa força o modelo a seguir o exemplo do Alecrim
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=False) # Não removemos os tokens especiais para análise posterior

def extract_triple(generated_text):
    """
    Versão atualizada para capturar o formato textual PERGUNTA/RESPOSTA
    e estruturar no formato exigido pelo professor (Instruction / Output).
    """
    
    pergunta_match = re.search(r"PERGUNTA:\s*(.*)", generated_text, re.IGNORECASE)
    resposta_match = re.search(r"RESPOSTA:\s*(.*)", generated_text, re.IGNORECASE)

    if pergunta_match and resposta_match:
        return {
            "Instruction": pergunta_match.group(1).strip(),
            "Output": resposta_match.group(1).strip()
        }
    return None



In [97]:
clean_chunks = [clean_text(c) for c in chunks if not is_bad_chunk(c)]
print(f"Chunks válidos: {len(clean_chunks)}")

Chunks válidos: 507


In [ ]:
triplas_causal = []
amostra_chunks = clean_chunks[:30]  # para demonstração, processamos apenas 30 chunks

for i, chunk in enumerate(amostra_chunks):
    prompt = PROMPT_CAUSAL.replace("{chunk}", chunk)
    try:
        result = run_model_direct(model, tokenizer, prompt)
        # O modelo causal pode repetir o prompt; removemos a parte inicial se existir
        if result.startswith(prompt):
            generated = result[len(prompt):].strip()
        else:
            generated = result
        triple = extract_triple(generated)
        if triple:  # só adiciona se extraiu algo real
            triplas_causal.append(triple)
            
        # if len(triple["output"]) > 20 and len(triple["instruction"]) > 5:
            # triplas_causal.append(triple)
            
        if i % 10 == 0:
            print(f"Processados {i}/{len(amostra_chunks)} chunks (causal)")
    except Exception as e:
        print(f"Erro no chunk {i}: {e}")

print(f"Triplas geradas (causal): {len(triplas_causal)}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processados 0/30 chunks (causal)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Processados 10/30 chunks (causal)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Processados 20/30 chunks (causal)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Triplas geradas (causal): 1


In [33]:
def clean_triple(triple):
    for key in triple:
        triple[key] = triple[key].strip()
    if triple["input"].lower() in ("none", "null", "n/a", ""):
        triple["input"] = ""
    if len(triple["output"]) < 20 or len(triple["instruction"]) < 5:
        return None
    return triple

def limpar_lista(triplas):
    return [t for t in (clean_triple(x) for x in triplas) if t is not None] # Filtra triplas inválidas após limpeza


triplas_causal = limpar_lista(triplas_causal)

print(f"Triplas causal após limpeza: {len(triplas_causal)}")

Triplas causal após limpeza: 44


In [34]:
def salvar_jsonl(triplas, nome_arquivo):
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        for t in triplas:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"Dataset salvo em {nome_arquivo}")

salvar_jsonl(triplas_causal, "dataset_causal_500chars.jsonl")

Dataset salvo em dataset_causal_500chars.jsonl


In [35]:
def mostrar_exemplo(triplas, titulo, n=3):
    print(f"\n=== {titulo} ===")
    for i, t in enumerate(triplas[:n], 1):
        print(f"\n--- Exemplo {i} ---")
        print(json.dumps(t, indent=2, ensure_ascii=False))

mostrar_exemplo(triplas_causal, "TinyLlama/TinyLlama-1.1B-Chat-v1.0")


=== TinyLlama/TinyLlama-1.1B-Chat-v1.0 ===

--- Exemplo 1 ---
{
  "instruction": "pergunta que um usuário faria sobre o texto",
  "input": "",
  "output": "exemplo de resposta para a pergunta"
}

--- Exemplo 2 ---
{
  "instruction": "Pergunta que um usuário faria sobre o texto",
  "input": "",
  "output": "O texto contém informações sobre a cura das plantas, o que significa que a inter-relação pode ser comuns entre a área de ciências biológicas e a área de ciências da saúde."
}

--- Exemplo 3 ---
{
  "instruction": "Explique o conteúdo do texto.",
  "input": "eberBiavatti:ProfessoraDoutoradoDepartamento Mayara Krasinski Caddah: Professora Doutora do Departa- MelissaCostaSantos:FarmacêuticadaPrefeituraMunicipalde FlorianópolisedaComissãodePráticasIntegrativaseComple- MuriloLeandroMarcos:Médicodefamíliaecomunidadedocen- PráticasIntegrativaseComplementaresdeFlorianópolis. ShirleyRosa:FarmacêuticacolaboradoradoHortodePlantas Prainha - Cláudia Schenya, Fernanda Nowak, Gisele Damian Antonio